In [ ]:
!pip install pyspark

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pandas as pd
from pyspark.sql.types import *

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
spark = SparkSession.builder.config("spark.driver.memory", "8g").getOrCreate()

In [ ]:
pandas_df = pd.read_excel("/content/drive/MyDrive/MAL_dados/anilist_anime_data_complete.xlsx", header=0)
anime_db = spark.createDataFrame(pandas_df)

In [ ]:
anime_db.show()
anime_db.count()


+----------+------+-------+--------------------+--------------------+--------------------------------------+--------------------+-----+--------+--------+--------------------+--------------+---------------+-------------+------------+-------------+-----------+------+----------+---------+--------+--------+--------+-------+---------------+----------+-----------+-------------+-----------+------------+--------------------+----------+---------------------+--------------------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+------------+---------+----------+----------+--------+--------------------+-----------+-------+--------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+-----------------------+------------------------+
|Unnamed: 0|    id|  i

20111

In [ ]:
anime_db.distinct().count()

20111

#Limpando o Dataframe

In [ ]:
anime_db=anime_db.drop('Unnamed','_c0','id','idMal',' description', 'siteUrl','bannerImage','title_native', 'title_userPreferred','hashtag','trailer_id','externalLinks','trailer_thumbnail','updatedAt','coverImage_extraLarge','coverImage_medium','coverImage_large','coverImage_color','rankings','streamingEpisodes','characters','reviews','stats_scoreDistribution','stats_statusDistribution')

In [ ]:
anime_db=anime_db.drop('recommendations','isLicensed','isLicensed','staff','source','trailer_site', 'nextAiringEpisode','trending','chapters','staff','airingSchedule','countryOfOrigin','relations','isLocked','')

In [ ]:
anime_db=anime_db.drop('isFavourite','synonyms','description','volumes')

In [ ]:
anime_db.show()
anime_db.distinct().count()

+----------+--------------------+--------------------+-----+--------+--------+--------------------+--------------+---------------+-------------+------------+-------------+-----------+------+----------+---------+--------+--------+-------+--------------------+--------------------+--------------------+------------+---------+----------+----------+-----------+-------+--------------------+
|Unnamed: 0|        title_romaji|       title_english| type|  format|  status|         description|startDate_year|startDate_month|startDate_day|endDate_year|endDate_month|endDate_day|season|seasonYear|seasonInt|episodes|duration|volumes|              genres|            synonyms|                tags|averageScore|meanScore|popularity|favourites|isFavourite|isAdult|             studios|
+----------+--------------------+--------------------+-----+--------+--------+--------------------+--------------+---------------+-------------+------------+-------------+-----------+------+----------+---------+--------+------

20111

#Tratando dados.

In [ ]:
frame_studio=anime_db.select('studios')

In [ ]:
frame_studio.show()
frame_studio.printSchema()
frame_studio.count()
frame_studio=frame_studio.fillna('Vazio', subset=['studios'])
frame_studio.show()
frame_studio.count()

+--------------------+
|             studios|
+--------------------+
|[{"id": 38511, "i...|
|[{"id": 17300, "i...|
|                  []|
|[{"id": 17293, "i...|
|[{"id": 42393, "i...|
|                  []|
|[{"id": 37342, "i...|
|[{"id": 25328, "i...|
|[{"id": 21032, "i...|
|[{"id": 16945, "i...|
|[{"id": 13548, "i...|
|[{"id": 15158, "i...|
|[{"id": 27936, "i...|
|[{"id": 10268, "i...|
|[{"id": 20525, "i...|
|[{"id": 41037, "i...|
|[{"id": 44777, "i...|
|[{"id": 17377, "i...|
|                  []|
|                  []|
+--------------------+
only showing top 20 rows
root
 |-- studios: string (nullable = true)

+--------------------+
|             studios|
+--------------------+
|[{"id": 38511, "i...|
|[{"id": 17300, "i...|
|                  []|
|[{"id": 17293, "i...|
|[{"id": 42393, "i...|
|                  []|
|[{"id": 37342, "i...|
|[{"id": 25328, "i...|
|[{"id": 21032, "i...|
|[{"id": 16945, "i...|
|[{"id": 13548, "i...|
|[{"id": 15158, "i...|
|[{"id": 27936, "i...|
|[{"id": 1

20111

#Separando os dados da coluna Studios, que possui dados semiestruturados em Json, então tive de separar os dados


In [ ]:


# O nome do estudio esta no campo NODE então fiz um inferschema para ele primeiro
node_schema = StructType([
    StructField('id', LongType(), True),
    StructField('name', StringType(), True),
    StructField('isAnimationStudio', BooleanType(), True)
])

#Agora definindo o inferschema da coluna como uma estrutura externa por campo Node
studio_object_schema = StructType([
    StructField('id', LongType(), True),
    StructField('isMain', BooleanType(), True),
    StructField('role', StringType(), True),
    StructField('node', node_schema, True) # 'node' is now a struct
])

# captando os dados do campo Studios e ai transformando no Array para
studios_array_schema = ArrayType(studio_object_schema)

# Agora aplicando From_json para transformar o campo em um Array
pedaco = frame_studio.withColumn('parsed_studios', from_json(col('studios'), studios_array_schema))

pedaco.show()

+--------------------+--------------------+
|             studios|      parsed_studios|
+--------------------+--------------------+
|[{"id": 38511, "i...|[{38511, false, N...|
|[{"id": 17300, "i...|[{17300, false, N...|
|                  []|                  []|
|[{"id": 17293, "i...|[{17293, true, NU...|
|[{"id": 42393, "i...|[{42393, true, NU...|
|                  []|                  []|
|[{"id": 37342, "i...|[{37342, true, NU...|
|[{"id": 25328, "i...|[{25328, false, N...|
|[{"id": 21032, "i...|[{21032, false, N...|
|[{"id": 16945, "i...|[{16945, true, NU...|
|[{"id": 13548, "i...|[{13548, true, NU...|
|[{"id": 15158, "i...|[{15158, false, N...|
|[{"id": 27936, "i...|[{27936, true, NU...|
|[{"id": 10268, "i...|[{10268, false, N...|
|[{"id": 20525, "i...|[{20525, true, NU...|
|[{"id": 41037, "i...|[{41037, false, N...|
|[{"id": 44777, "i...|[{44777, true, NU...|
|[{"id": 17377, "i...|[{17377, true, NU...|
|                  []|                  []|
|                  []|          

In [ ]:
anime_db.printSchema()

root
 |-- Unnamed: 0: long (nullable = true)
 |-- title_romaji: string (nullable = true)
 |-- title_english: string (nullable = true)
 |-- type: string (nullable = true)
 |-- format: string (nullable = true)
 |-- status: string (nullable = true)
 |-- startDate_year: long (nullable = true)
 |-- startDate_month: double (nullable = true)
 |-- startDate_day: double (nullable = true)
 |-- endDate_year: double (nullable = true)
 |-- endDate_month: double (nullable = true)
 |-- endDate_day: double (nullable = true)
 |-- season: string (nullable = true)
 |-- seasonYear: double (nullable = true)
 |-- seasonInt: double (nullable = true)
 |-- episodes: double (nullable = true)
 |-- duration: double (nullable = true)
 |-- genres: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- averageScore: double (nullable = true)
 |-- meanScore: double (nullable = true)
 |-- popularity: long (nullable = true)
 |-- favourites: long (nullable = true)
 |-- isAdult: boolean (nullable = true)
 |-- s

### Extraindo e Filtrando o nome do estúdio de Animação
(Com auxilio de IA)

In [ ]:
from pyspark.sql.functions import col, from_json, when, lit, udf
from pyspark.sql.types import StringType

# Reusing schemas defined previously in cell BXNSWLY9PzJv
# node_schema, studio_object_schema, studios_array_schema

# 1. Parse o JSON da coluna 'studios' no DataFrame principal
anime_db = anime_db.withColumn(
    'parsed_studios',
    from_json(col('studios'), studios_array_schema)
)

# Define a UDF to extract the principal animation studio name
def get_principal_animation_studio(studios_list):
    if not studios_list:
        return 'Nulo'

    main_animation_studio_name = None
    first_animation_studio_name = None

    for studio_obj in studios_list:
        # Check if 'node' and 'isAnimationStudio' exist and is True
        # Accessing nested fields of a Row object
        if studio_obj and 'node' in studio_obj and studio_obj['node'] and 'isAnimationStudio' in studio_obj['node'] and studio_obj['node']['isAnimationStudio']:
            if 'isMain' in studio_obj and studio_obj['isMain']:
                # If a main animation studio is found, return its name immediately
                return studio_obj['node']['name']
            if first_animation_studio_name is None:
                # Store the first animation studio found (if no main one is found later)
                first_animation_studio_name = studio_obj['node']['name']

    # If no main animation studio was found, return the first animation studio's name
    if first_animation_studio_name:
        return first_animation_studio_name

    # If no animation studios were found at all
    return 'Sem estudio'

# Register the UDF
get_principal_animation_studio_udf = udf(get_principal_animation_studio, StringType())

# Apply the UDF to create the 'Estudio_Principal' column
anime_db = anime_db.withColumn(
    'Estudio',
    get_principal_animation_studio_udf(col('parsed_studios'))
)

# Remova as colunas intermediárias 'parsed_studios' e 'animation_studios_list'
anime_db = anime_db.drop('parsed_studios', 'studios')

# Exiba o DataFrame atualizado para verificar a nova coluna
anime_db.select('title_romaji', 'Estudio').show(truncate=False)
anime_db.count()

+-------------------------------------------------------------------------+-----------------------+
|title_romaji                                                             |Estudio                |
+-------------------------------------------------------------------------+-----------------------+
|Zutto Tomodachi                                                          |Sem estudio            |
|Zutto Suki Datta                                                         |Queen Bee              |
|Zutto Mae kara Suki deshita.: Kokuhaku Jikkou Iinkai - Kinyoubi no Ohayou|Nulo                   |
|Zutto Mae kara Suki deshita.: Kokuhaku Jikkou Iinkai                     |Qualia Animation       |
|Zutaboro Reijou wa Ane no Moto Konyakusha ni Dekiai Sareru               |LandQ studios          |
|Zurui Maboroshi                                                          |Nulo                   |
|Zuoshou Shanglan 2                                                       |Heart & Soul Animation |


20111

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, BooleanType, ArrayType
from pyspark.sql.functions import col, from_json, concat_ws, array_distinct, expr # Import expr for transform if needed. Col.transform is available in recent versions.

# Define schema for a single tag object
# Renaming for clarity as it's for 'tags' column, not 'studios'
tag_object_schema = StructType([
    StructField('id', LongType(), True),
    StructField('name', StringType(), True),
    StructField('description', StringType(), True),
    StructField('category', StringType(), True),
    StructField('rank', LongType(), True),
    StructField('isGeneralSpoiler', BooleanType(), True),
    StructField('isMediaSpoiler', BooleanType(), True),
    StructField('isAdult', BooleanType(), True)
])

# Define the overall schema for the 'tags' column, which is an array of tag objects
tags_array_schema = ArrayType(tag_object_schema)

# Parse the JSON string in the 'tags' column into a structured array
anime_db = anime_db.withColumn('parsed_tags', from_json(col('tags'), tags_array_schema))

# Extract the 'name' from each tag object in the 'parsed_tags' array
# and collect them into a new array column 'tag_names_array'.
# Use a higher-order function like transform (available in Spark 3.1+)
# to apply a transformation to each element of the array.
# For older Spark versions, a UDF would be necessary.
# Colab typically runs on Spark 3.x, so transform should be available.

anime_db = anime_db.withColumn(
    'tag_names_array',
    expr("transform(parsed_tags, x -> x.name)") # Using expr with transform
)

# Convert the array of tag names into a comma-separated string
# Use array_distinct to ensure unique tags if desired.
anime_db = anime_db.withColumn('tags', concat_ws(', ', array_distinct(col('tag_names_array'))))

# Drop the intermediate parsing columns
anime_db = anime_db.drop('parsed_tags', 'tag_names_array')

# Show the updated DataFrame with the new 'tags' column
anime_db.select('title_romaji', 'tags').show(truncate=False)



+-------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|title_romaji                                                             |tags                                                                                                                                                                            |
+-------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Zutto Tomodachi                                                          |                                                                                                                                                                      

In [ ]:
anime_db.show()
anime_db.count()

+----------+--------------------+--------------------+-----+--------+--------+--------------+---------------+-------------+------------+-------------+-----------+------+----------+---------+--------+--------+--------------------+--------------------+------------+---------+----------+----------+-------+--------------------+
|Unnamed: 0|        title_romaji|       title_english| type|  format|  status|startDate_year|startDate_month|startDate_day|endDate_year|endDate_month|endDate_day|season|seasonYear|seasonInt|episodes|duration|              genres|                tags|averageScore|meanScore|popularity|favourites|isAdult|             Estudio|
+----------+--------------------+--------------------+-----+--------+--------+--------------+---------------+-------------+------------+-------------+-----------+------+----------+---------+--------+--------+--------------------+--------------------+------------+---------+----------+----------+-------+--------------------+
|         0|     Zutto To

20111

In [ ]:
anime_db=anime_db.withColumn("Tags_1", get(split(col("tags"), ","),0))
anime_db=anime_db.withColumn("Tags_2", get(split(col("tags"), ","),1))
anime_db=anime_db.withColumn("Tags_3", get(split(col("tags"), ","),2))
anime_db=anime_db.withColumn("Tags_4", get(split(col("tags"), ","),3))
anime_db=anime_db.withColumn("Tags_5", get(split(col("tags"), ","),4))
anime_db=anime_db.withColumn("Tags_6", get(split(col("tags"), ","),5))
anime_db=anime_db.withColumn("Tags_7", get(split(col("tags"), ","),6))
anime_db=anime_db.withColumn("Tags_8", get(split(col("tags"), ","),7))
anime_db=anime_db.withColumn("Tags_9", get(split(col("tags"), ","),8))
anime_db=anime_db.withColumn("Tags_10", get(split(col("tags"), ","),9))
anime_db=anime_db.withColumn("Tags_11", get(split(col("tags"), ","),10))
anime_db=anime_db.withColumn("Tags_12", get(split(col("tags"), ","),11))
anime_db=anime_db.withColumn("Tags_13", get(split(col("tags"), ","),12))
anime_db=anime_db.withColumn("Tags_14", get(split(col("tags"), ","),13))
anime_db=anime_db.withColumn("Tags_15", get(split(col("tags"), ","),14))
anime_db=anime_db.withColumn("Tags_16", get(split(col("tags"), ","),15))
anime_db=anime_db.withColumn("Tags_17", get(split(col("tags"), ","),16))
anime_db=anime_db.withColumn("Tags_18", get(split(col("tags"), ","),17))
anime_db=anime_db.withColumn("Tags_19", get(split(col("tags"), ","),18))
anime_db=anime_db.withColumn("Tags_20", get(split(col("tags"), ","),19))
anime_db=anime_db.withColumn("Tags_21", get(split(col("tags"), ","),20))
anime_db=anime_db.withColumn("Tags_22", get(split(col("tags"), ","),21))
anime_db=anime_db.withColumn("Tags_23", get(split(col("tags"), ","),22))
anime_db=anime_db.withColumn("Tags_24", get(split(col("tags"), ","),23))
anime_db=anime_db.withColumn("Tags_25", get(split(col("tags"), ","),24))
anime_db=anime_db.withColumn("Tags_26", get(split(col("tags"), ","),25))
anime_db=anime_db.withColumn("Tags_27", get(split(col("tags"), ","),26))
anime_db=anime_db.withColumn("Tags_28", get(split(col("tags"), ","),27))
anime_db=anime_db.withColumn("Tags_29", get(split(col("tags"), ","),28))
anime_db=anime_db.withColumn("Tags_30", get(split(col("tags"), ","),29))
anime_db=anime_db.withColumn("Tags_31", get(split(col("tags"), ","),30))
anime_db=anime_db.withColumn("Tags_32", get(split(col("tags"), ","),31))
anime_db=anime_db.withColumn("Tags_33", get(split(col("tags"), ","),32))
anime_db=anime_db.withColumn("Tags_34", get(split(col("tags"), ","),33))
anime_db=anime_db.withColumn("Tags_35", get(split(col("tags"), ","),34))
anime_db=anime_db.withColumn("Tags_36", get(split(col("tags"), ","),35))
anime_db=anime_db.withColumn("Tags_37", get(split(col("tags"), ","),36))
anime_db=anime_db.withColumn("Tags_38", get(split(col("tags"), ","),37))
anime_db=anime_db.withColumn("Tags_39", get(split(col("tags"), ","),38))
anime_db=anime_db.withColumn("Tags_40", get(split(col("tags"), ","),39))
anime_db=anime_db.withColumn("Tags_41", get(split(col("tags"), ","),40))
anime_db=anime_db.withColumn("Tags_42", get(split(col("tags"), ","),41))
anime_db=anime_db.withColumn("Tags_43", get(split(col("tags"), ","),42))
anime_db=anime_db.withColumn("Tags_44", get(split(col("tags"), ","),43))
anime_db=anime_db.withColumn("Tags_45", get(split(col("tags"), ","),44))
anime_db=anime_db.withColumn("Tags_46", get(split(col("tags"), ","),45))
anime_db=anime_db.withColumn("Tags_47", get(split(col("tags"), ","),46))
anime_db=anime_db.withColumn("Tags_48", get(split(col("tags"), ","),47))
anime_db=anime_db.withColumn("Tags_49", get(split(col("tags"), ","),48))
anime_db=anime_db.withColumn("Tags_50", get(split(col("tags"), ","),49))
anime_db=anime_db.withColumn("Tags_51", get(split(col("tags"), ","),50))
anime_db=anime_db.withColumn("Tags_52", get(split(col("tags"), ","),51))
anime_db=anime_db.withColumn("Tags_53", get(split(col("tags"), ","),52))
anime_db=anime_db.withColumn("Tags_54", get(split(col("tags"), ","),53))
anime_db=anime_db.withColumn("Tags_55", get(split(col("tags"), ","),54))
anime_db=anime_db.withColumn("Tags_56", get(split(col("tags"), ","),55))
anime_db=anime_db.withColumn("Tags_57", get(split(col("tags"), ","),56))
anime_db=anime_db.withColumn("Tags_58", get(split(col("tags"), ","),57))
anime_db=anime_db.withColumn("Tags_59", get(split(col("tags"), ","),58))
anime_db=anime_db.withColumn("Tags_60", get(split(col("tags"), ","),59))
anime_db=anime_db.withColumn("Tags_61", get(split(col("tags"), ","),60))
anime_db=anime_db.withColumn("Tags_62", get(split(col("tags"), ","),61))
anime_db=anime_db.withColumn("Tags_63", get(split(col("tags"), ","),62))
anime_db=anime_db.withColumn("Tags_64", get(split(col("tags"), ","),63))
anime_db=anime_db.withColumn("Tags_65", get(split(col("tags"), ","),64))
anime_db=anime_db.withColumn("Tags_66", get(split(col("tags"), ","),65))
anime_db=anime_db.withColumn("Tags_67", get(split(col("tags"), ","),66))
anime_db=anime_db.withColumn("Tags_68", get(split(col("tags"), ","),67))
anime_db=anime_db.withColumn("Tags_69", get(split(col("tags"), ","),68))
anime_db=anime_db.withColumn("Tags_70", get(split(col("tags"), ","),69))




In [ ]:
anime_db=anime_db.withColumnRenamed('title_romaji','Titulo')
anime_db=anime_db.withColumnRenamed('title_english','Titulo_Ingles')
anime_db=anime_db.withColumnRenamed('type','Tipo')
anime_db=anime_db.withColumnRenamed('status','Status')
anime_db=anime_db.withColumnRenamed('duration','Duracao')
anime_db=anime_db.withColumnRenamed('format', 'formato')
anime_db=anime_db.withColumnRenamed('popularity','Popularidade')
anime_db=anime_db.withColumnRenamed('season','Temporada')
anime_db=anime_db.withColumnRenamed('seasonYear','Ano da Temporada')

In [ ]:
anime_db=anime_db.withColumnRenamed('startDate_month','Mês de estreia')
anime_db=anime_db.withColumnRenamed('startDate_year','Ano de estreia')
anime_db=anime_db.withColumnRenamed('startDate_day','Dia de estreia')
anime_db=anime_db.withColumnRenamed('endDate_month','Mês de encerramento')
anime_db=anime_db.withColumnRenamed('endDate_year','Ano de encerramento')
anime_db=anime_db.withColumnRenamed('endDate_day','Dia de encerramento')
anime_db=anime_db.withColumnRenamed('averageScore','Nota Media de Notas')
anime_db=anime_db.withColumnRenamed('meanScore','Media de notas')
anime_db=anime_db.withColumnRenamed('favourites','Quantidade de Votos')
anime_db=anime_db.withColumnRenamed('episodes','Quantidade de Episodios')

anime_db=anime_db.withColumnRenamed('isAdult','É Adulto')

In [ ]:
anime_db=anime_db.fillna(0)

In [ ]:
#Mais um campo Json para tratar
genres_schema = ArrayType(StringType())
anime_db = anime_db.withColumn('Genero', from_json(col('genres'), genres_schema))


In [ ]:
# pegar a coluna, transformada em Array, e juntar numa string só, usando o comando Concat_ws, separando pela virgula
anime_db = anime_db.withColumn('Genero divição', concat_ws(',', col('Genero')))

In [ ]:


anime_db=anime_db.withColumn('Genero dividir_1', get(split(col('Genero divição'), ','), 0))
anime_db=anime_db.withColumn('Genero dividir_2', get(split(col('Genero divição'), ','), 1))
anime_db=anime_db.withColumn('Genero dividir_3', get(split(col('Genero divição'), ','), 2))
anime_db=anime_db.withColumn('Genero dividir_4', get(split(col('Genero divição'), ','), 3))
anime_db=anime_db.withColumn('Genero dividir_5', get(split(col('Genero divição'), ','), 4))
anime_db=anime_db.withColumn('Genero dividir_6', get(split(col('Genero divição'), ','), 5))
anime_db=anime_db.withColumn('Genero dividir_7', get(split(col('Genero divição'), ','), 6))
anime_db.show()

+----------+--------------------+--------------------+-----+--------+--------+--------------+--------------+--------------+-------------------+-------------------+-------------------+---------+----------------+---------+-----------------------+-------+--------------------+--------------------+-------------------+--------------+------------+-------------------+--------+--------------------+-------------+--------------------+--------------------+--------------------+-------------------+--------------+-------------------+------------------+-----------+--------------+-------+---------+-------+-------+----------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-

In [ ]:
temporada_pt={'WINTER':'Inverno','SPRING':'Primavera','SUMMER':'Verão','FALL':'Outono', 'NaN':'Nulo'}
Tipo_pt={'TV':'TV','MOVIE':'Filme','SPECIAL':'Especial','OVA':'OVA','ONA':'Streaming/Internet','MUSIC':'Música'}
status_pt={'FINISHED':'Finalizado','RELEASING':'Lançando','NOT_YET_RELEASED':'Não lançado','CANCELLED':'Cancelado','HIATUS':'Hiato'}
anime_db = anime_db.replace(to_replace=temporada_pt, subset=['Temporada'])
anime_db = anime_db.replace(to_replace=Tipo_pt, subset=['formato'])
anime_db = anime_db.replace(to_replace=status_pt, subset=['Status'])

In [ ]:
anime_db=anime_db.drop('Genero','Genero divição','genres','Unnamed: 0')
anime_db=anime_db.dropna(subset=['Titulo'])

In [ ]:
anime_db.describe()

DataFrame[summary: string, Titulo: string, Titulo_Ingles: string, Tipo: string, formato: string, Status: string, Ano de estreia: string, Mês de estreia: string, Dia de estreia: string, Ano de encerramento: string, Mês de encerramento: string, Dia de encerramento: string, Temporada: string, Ano da Temporada: string, seasonInt: string, Quantidade de Episodios: string, Duracao: string, tags: string, Nota Media de Notas: string, Media de notas: string, Popularidade: string, Quantidade de Votos: string, Estudio: string, Tags_1: string, Tags_2: string, Tags_3: string, Tags_4: string, Tags_5: string, Tags_6: string, Tags_7: string, Tags_8: string, Tags_9: string, Tags_10: string, Tags_11: string, Tags_12: string, Tags_13: string, Tags_14: string, Tags_15: string, Tags_16: string, Tags_17: string, Tags_18: string, Tags_19: string, Tags_20: string, Tags_21: string, Tags_22: string, Tags_23: string, Tags_24: string, Tags_25: string, Tags_26: string, Tags_27: string, Tags_28: string, Tags_29: str

In [ ]:
anime_db = anime_db.drop('Genero')
fatura=anime_db.select('genero dividir_1').distinct()
fatura.head(20)
conta={ 'Romance':1, 'Adventure':2, 'Comedy':3, 'Sports':4, 'Drama':5, 'Hentai':6, 'Mahou Shoujo':7, 'Fantasy':8, 'Mecha':9, 'Mystery':10, 'Supernatural':11, 'Music':12, 'Slice of Life':13, 'Ecchi':14, 'Psychological':15, 'Horror':16, 'Action':17, 'Sci-Fi':18, 'Thriller':19, ' ':0}

In [ ]:
#Fazendo contas com a variavel

def fazendo_calculos(column_name, mapping_dict):
    exp_mappin= lit(None) # Iniciar com valor nulo
    for gen_str, num_val in mapping_dict.items():
        exp_mappin = when(col(column_name) == gen_str, num_val).otherwise(exp_mappin)
    return exp_mappin


In [ ]:
anime_db = anime_db.withColumn('Gen_nun_1', fazendo_calculos('Genero dividir_1', conta))
anime_db = anime_db.withColumn('Gen_nun_2', fazendo_calculos('Genero dividir_2', conta))
anime_db = anime_db.withColumn('Gen_nun_3', fazendo_calculos('Genero dividir_3', conta))
anime_db = anime_db.withColumn('Gen_nun_4', fazendo_calculos('Genero dividir_4', conta))
anime_db = anime_db.withColumn('Gen_nun_5', fazendo_calculos('Genero dividir_5', conta))
anime_db = anime_db.withColumn('Gen_nun_6', fazendo_calculos('Genero dividir_6', conta))
anime_db = anime_db.withColumn('Gen_nun_7', fazendo_calculos('Genero dividir_7', conta))

In [ ]:
anime_db.show()

+--------------------+--------------------+-----+------------------+----------+--------------+--------------+--------------+-------------------+-------------------+-------------------+---------+----------------+---------+-----------------------+-------+--------------------+-------------------+--------------+------------+-------------------+--------+--------------------+-------------+--------------------+--------------------+--------------------+-------------------+--------------+-------------------+------------------+-----------+--------------+-------+---------+-------+-------+----------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-----

In [ ]:
anime_db = anime_db.withColumn('Mês de estreia', col('Mês de estreia').cast(IntegerType()))
anime_db = anime_db.withColumn('Ano de estreia', col('Ano de estreia').cast(IntegerType()))
anime_db = anime_db.withColumn('Dia de estreia', col('Dia de estreia').cast(IntegerType()))
anime_db = anime_db.withColumn('Mês de encerramento', col('Mês de encerramento').cast(IntegerType()))
anime_db = anime_db.withColumn('Ano de encerramento', col('Ano de encerramento').cast(IntegerType()))
anime_db = anime_db.withColumn('Dia de encerramento', col('Dia de encerramento').cast(IntegerType()))
anime_db = anime_db.withColumn('Ano da Temporada', col('Ano da Temporada').cast(IntegerType()))
anime_db = anime_db.withColumn('Quantidade de Episodios', col('Quantidade de Episodios').cast(IntegerType()))
anime_db = anime_db.withColumn('Popularidade', col('Popularidade').cast(IntegerType()))
anime_db = anime_db.withColumn('Quantidade de Votos', col('Quantidade de Votos').cast(IntegerType()))
anime_db = anime_db.withColumn('Media de notas', col('Media de notas').cast(IntegerType()))
anime_db = anime_db.withColumn('Nota Media de Notas', col('Nota Media de Notas').cast(IntegerType()))
anime_db = anime_db.withColumn('Duracao', col('Duracao').cast(IntegerType()))
anime_db

DataFrame[Titulo: string, Titulo_Ingles: string, Tipo: string, formato: string, Status: string, Ano de estreia: int, Mês de estreia: int, Dia de estreia: int, Ano de encerramento: int, Mês de encerramento: int, Dia de encerramento: int, Temporada: string, Ano da Temporada: int, seasonInt: double, Quantidade de Episodios: int, Duracao: int, tags: string, Nota Media de Notas: int, Media de notas: int, Popularidade: int, Quantidade de Votos: int, É Adulto: boolean, Estudio: string, Tags_1: string, Tags_2: string, Tags_3: string, Tags_4: string, Tags_5: string, Tags_6: string, Tags_7: string, Tags_8: string, Tags_9: string, Tags_10: string, Tags_11: string, Tags_12: string, Tags_13: string, Tags_14: string, Tags_15: string, Tags_16: string, Tags_17: string, Tags_18: string, Tags_19: string, Tags_20: string, Tags_21: string, Tags_22: string, Tags_23: string, Tags_24: string, Tags_25: string, Tags_26: string, Tags_27: string, Tags_28: string, Tags_29: string, Tags_30: string, Tags_31: string

In [ ]:
anime_db.coalesce(1).write.csv('/content/drive/MyDrive/MAL_dados/anime_db.csv/anime_db/mal.csv', header=True, mode='overwrite')
